# Experiment Report Dashboard

This notebook is a dashboard for reviewing **all saved experiment reports**. Its purpose is to help you see:

- which experiments have been run
- how each model performed inside each experiment
- how the same model changed from experiment to experiment
- when a model became stronger or weaker
- which experiment configuration produced the best result for a chosen model

It is designed to work both with the global summary CSV files from `src/main.py` and with individual experiment folders created manually from the playground notebook.


## 1. Setup

We load the output directory, set display options, and define small helpers for reading experiment reports safely.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'task-house-prices':
    NOTEBOOK_DIR = Path('/Users/maksymponomarenko/Documents/ai-engineering-kaggle-portfolio/courses/01-machine-learning-with-python/tasks/task-house-prices')

OUTPUTS_DIR = NOTEBOOK_DIR / 'outputs' / 'experiments'
OUTPUTS_DIR

In [ ]:
def load_json_if_exists(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding='utf-8'))


def discover_experiment_dirs(outputs_dir: Path) -> list[Path]:
    if not outputs_dir.exists():
        return []
    return sorted(
        [path for path in outputs_dir.iterdir() if path.is_dir() and (path / 'model_comparison.csv').exists()],
        key=lambda path: path.name,
    )


def build_summary_from_experiment_dirs(outputs_dir: Path) -> pd.DataFrame:
    rows: list[pd.DataFrame] = []
    for experiment_dir in discover_experiment_dirs(outputs_dir):
        comparison_path = experiment_dir / 'model_comparison.csv'
        comparison_df = pd.read_csv(comparison_path)
        definition = load_json_if_exists(experiment_dir / 'experiment_definition.json')
        if comparison_df.empty:
            continue
        best_model_name = comparison_df.sort_values(['rmse_mean', 'mae_mean']).iloc[0]['model_name']
        comparison_df = comparison_df.copy()
        comparison_df.insert(0, 'experiment_name', experiment_dir.name)
        comparison_df.insert(1, 'experiment_description', definition.get('description', ''))
        comparison_df['is_best_model'] = comparison_df['model_name'].eq(best_model_name)
        comparison_df['best_model_name'] = best_model_name
        comparison_df['experiment_dir'] = str(experiment_dir)
        rows.append(comparison_df)

    if not rows:
        return pd.DataFrame()

    result = pd.concat(rows, ignore_index=True)
    return result.sort_values(['experiment_name', 'rmse_mean', 'mae_mean']).reset_index(drop=True)


def load_global_or_local_summary(outputs_dir: Path) -> pd.DataFrame:
    global_summary_path = outputs_dir / 'experiment_summary_all_models.csv'
    if global_summary_path.exists():
        return pd.read_csv(global_summary_path)
    return build_summary_from_experiment_dirs(outputs_dir)


def make_metric_pivot(summary_df: pd.DataFrame, metric_name: str) -> pd.DataFrame:
    if summary_df.empty:
        return pd.DataFrame()
    return (
        summary_df.pivot(index='experiment_name', columns='model_name', values=metric_name)
        .sort_index()
        .reset_index()
    )


def style_best_per_experiment(df: pd.DataFrame):
    if df.empty:
        return df.style

    def highlight_best(row: pd.Series) -> list[str]:
        is_best = bool(row.get('is_best_model', False))
        return ['background-color: #d9f2d9' if is_best else '' for _ in row]

    return df.style.apply(highlight_best, axis=1)


## 2. Discover available experiment outputs

This section shows which experiment folders are currently available. If you only ran one playground experiment so far, that is still fine: the notebook will work on that subset.


In [ ]:
experiment_dirs = discover_experiment_dirs(OUTPUTS_DIR)
print(f'Found {len(experiment_dirs)} experiment directories.')
pd.DataFrame({'experiment_dir': [str(path) for path in experiment_dirs]})

In [ ]:
summary_df = load_global_or_local_summary(OUTPUTS_DIR)
print('Summary shape:', summary_df.shape)
summary_df.head(20)

## 3. All models across all experiments

This is the main table to review model dynamics. Each row is one model inside one experiment.

Use it to answer questions like:

- how did `ridge` change when I added ordinal features?
- when did `random_forest` overtake the linear models?
- did a feature engineering step help every model or only one family?


In [ ]:
if summary_df.empty:
    display(pd.DataFrame({'message': ['No experiment outputs found yet. Run src/main.py or a playground experiment first.']}))
else:
    display(style_best_per_experiment(summary_df))

In [ ]:
if not summary_df.empty:
    summary_df.groupby('model_name')[['rmse_mean', 'mae_mean', 'r2_mean']].agg(['mean', 'min', 'max']).round(4)
else:
    pd.DataFrame()

## 4. Best model by experiment

This compact view helps you see which model won inside each experiment. It is useful, but it should not replace the full table above, because your main goal is to understand **how each model changes** across experiments.


In [ ]:
best_models_df = summary_df.loc[summary_df['is_best_model']].sort_values('rmse_mean').reset_index(drop=True) if not summary_df.empty else pd.DataFrame()
best_models_df

## 5. Pivot tables for side-by-side comparison

These pivots are the fastest way to compare a single metric across experiments and models.

- For `RMSE` and `MAE`, lower is better.
- For `R²`, higher is better.


In [ ]:
pivot_rmse = make_metric_pivot(summary_df, 'rmse_mean')
pivot_rmse

In [ ]:
pivot_mae = make_metric_pivot(summary_df, 'mae_mean')
pivot_mae

In [ ]:
pivot_r2 = make_metric_pivot(summary_df, 'r2_mean')
pivot_r2

## 6. Heatmaps

Heatmaps make it easier to spot patterns quickly, especially when you have several experiments.


In [ ]:
def plot_metric_heatmap(summary_df: pd.DataFrame, metric_name: str, title: str, higher_is_better: bool) -> None:
    if summary_df.empty:
        print('No summary data available.')
        return

    heatmap_df = summary_df.pivot(index='experiment_name', columns='model_name', values=metric_name)
    plt.figure(figsize=(10, max(4, len(heatmap_df) * 0.6)))
    cmap = 'YlGn' if higher_is_better else 'YlOrRd_r'
    sns.heatmap(heatmap_df, annot=True, fmt='.4f', cmap=cmap)
    plt.title(title)
    plt.tight_layout()

In [ ]:
plot_metric_heatmap(summary_df, 'rmse_mean', 'RMSE by experiment and model', higher_is_better=False)

In [ ]:
plot_metric_heatmap(summary_df, 'mae_mean', 'MAE by experiment and model', higher_is_better=False)

In [ ]:
plot_metric_heatmap(summary_df, 'r2_mean', 'R² by experiment and model', higher_is_better=True)

## 7. Model trajectories across experiments

Sometimes the most useful question is not “what was the best model once?” but “how did this model behave as my feature engineering changed?”


In [ ]:
def plot_model_metric_over_experiments(summary_df: pd.DataFrame, metric_name: str, title: str) -> None:
    if summary_df.empty:
        print('No summary data available.')
        return

    plot_df = summary_df.copy()
    experiment_order = (
        plot_df.groupby('experiment_name')['rmse_mean']
        .min()
        .sort_values()
        .index
        .tolist()
    )
    plot_df['experiment_name'] = pd.Categorical(plot_df['experiment_name'], categories=experiment_order, ordered=True)
    plot_df = plot_df.sort_values(['experiment_name', 'model_name'])

    plt.figure(figsize=(12, 6))
    sns.lineplot(data=plot_df, x='experiment_name', y=metric_name, hue='model_name', marker='o')
    plt.title(title)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

In [ ]:
plot_model_metric_over_experiments(summary_df, 'rmse_mean', 'RMSE trajectory by model across experiments')

In [ ]:
plot_model_metric_over_experiments(summary_df, 'mae_mean', 'MAE trajectory by model across experiments')

In [ ]:
plot_model_metric_over_experiments(summary_df, 'r2_mean', 'R² trajectory by model across experiments')

## 8. Drill down into one model

Choose a model name and inspect its performance across all experiments. This is often the most helpful view when you are trying to improve a specific model family, such as `ridge` or `random_forest`.


In [ ]:
selected_model = 'ridge'
selected_model

In [ ]:
model_history_df = (
    summary_df.loc[summary_df['model_name'] == selected_model]
    .sort_values('rmse_mean')
    .reset_index(drop=True)
    if not summary_df.empty else pd.DataFrame()
)
model_history_df

In [ ]:
if not model_history_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.barplot(data=model_history_df, x='rmse_mean', y='experiment_name', ax=axes[0])
    axes[0].set_title(f'{selected_model}: RMSE by experiment')

    sns.barplot(data=model_history_df, x='mae_mean', y='experiment_name', ax=axes[1])
    axes[1].set_title(f'{selected_model}: MAE by experiment')

    sns.barplot(data=model_history_df, x='r2_mean', y='experiment_name', ax=axes[2])
    axes[2].set_title(f'{selected_model}: R² by experiment')

    plt.tight_layout()
else:
    print('No rows found for the selected model.')

## 9. Drill down into one experiment

Choose an experiment and inspect everything saved for it: model comparison, definition, holdout metrics, and permutation importance.


In [ ]:
selected_experiment = summary_df['experiment_name'].iloc[0] if not summary_df.empty else None
selected_experiment

In [ ]:
if selected_experiment is None:
    experiment_dir = None
    print('No experiment selected.')
else:
    experiment_dir = OUTPUTS_DIR / selected_experiment
    print(experiment_dir)
    
experiment_dir

In [ ]:
if experiment_dir is not None and (experiment_dir / 'experiment_definition.json').exists():
    experiment_definition = load_json_if_exists(experiment_dir / 'experiment_definition.json')
    experiment_definition
else:
    {}

In [ ]:
if experiment_dir is not None and (experiment_dir / 'model_comparison.csv').exists():
    experiment_model_comparison = pd.read_csv(experiment_dir / 'model_comparison.csv')
    experiment_model_comparison
else:
    pd.DataFrame()

In [ ]:
if experiment_dir is not None and (experiment_dir / 'holdout_metrics.json').exists():
    pd.DataFrame([load_json_if_exists(experiment_dir / 'holdout_metrics.json')])
else:
    pd.DataFrame()

In [ ]:
if experiment_dir is not None and (experiment_dir / 'permutation_importance.csv').exists():
    experiment_importance = pd.read_csv(experiment_dir / 'permutation_importance.csv')
    experiment_importance.head(20)
else:
    pd.DataFrame()

In [ ]:
if experiment_dir is not None and (experiment_dir / 'permutation_importance.csv').exists():
    plot_df = pd.read_csv(experiment_dir / 'permutation_importance.csv').head(20).sort_values('importance_mean')
    plt.figure(figsize=(10, 8))
    plt.barh(plot_df['feature'], plot_df['importance_mean'])
    plt.title(f'Top permutation importances: {selected_experiment}')
    plt.tight_layout()
else:
    print('No permutation importance file found for the selected experiment.')

## 10. Compare two experiments directly

This section helps when you want to compare a “before” and “after” experiment, for example before and after adding ordinal features.


In [ ]:
available_experiments = summary_df['experiment_name'].drop_duplicates().tolist() if not summary_df.empty else []
experiment_a = available_experiments[0] if len(available_experiments) >= 1 else None
experiment_b = available_experiments[1] if len(available_experiments) >= 2 else experiment_a
experiment_a, experiment_b

In [ ]:
if experiment_a is not None and experiment_b is not None and not summary_df.empty:
    compare_ab = (
        summary_df.loc[summary_df['experiment_name'].isin([experiment_a, experiment_b]), ['experiment_name', 'model_name', 'rmse_mean', 'mae_mean', 'r2_mean']]
        .sort_values(['model_name', 'experiment_name'])
        .reset_index(drop=True)
    )
    display(compare_ab)
else:
    print('Need at least one experiment to compare.')

In [ ]:
if experiment_a is not None and experiment_b is not None and not summary_df.empty:
    comparison_pivot = (
        summary_df.loc[summary_df['experiment_name'].isin([experiment_a, experiment_b]), ['experiment_name', 'model_name', 'rmse_mean', 'mae_mean', 'r2_mean']]
        .pivot(index='model_name', columns='experiment_name', values='rmse_mean')
    )
    comparison_pivot
else:
    pd.DataFrame()

## 11. Practical interpretation checklist

When you review this dashboard, try to answer these questions:

- Did a new feature group help every model or only one family?
- Which model is most stable across experiments?
- Which model benefits most from richer preprocessing?
- Is the “best model” changing because the experiment genuinely improved, or because one model is more sensitive to the change?
- Which experiment gave the strongest version of the model you actually care about improving?


In [ ]:
if not summary_df.empty:
    model_wins = summary_df.loc[summary_df['is_best_model'], 'model_name'].value_counts().to_frame(name='experiment_wins')
    model_wins
else:
    pd.DataFrame()

In [ ]:
if not summary_df.empty:
    best_rmse_row = summary_df.sort_values('rmse_mean').iloc[0]
    best_r2_row = summary_df.sort_values('r2_mean', ascending=False).iloc[0]
    pd.DataFrame([
        {'metric_focus': 'lowest_rmse', **best_rmse_row.to_dict()},
        {'metric_focus': 'highest_r2', **best_r2_row.to_dict()},
    ])
else:
    pd.DataFrame()